# 🚀 Employment Prediction Hackathon — R Starter Kit
Welcome! This notebook walks you from raw data to your first leaderboard submission, step by step.

**The goal:** for each person in Round 9, predict the *probability* that they are **employed (1)** vs **unemployed (0)**.
Your submissions are scored on **AUC**, so always submit a probability between 0 and 1 — never just a hard 0/1.

## 🛠️ Step 1: Load tools and data


In [1]:
# install.packages(c('scales','pROC'))  # uncomment on first run if needed
suppressMessages({
  library(dplyr)
  library(ggplot2)
  library(scales)
})

# Load the competition datasets (adjust the paths to match your Kaggle input folder)
data_dir <- '/kaggle/input/competitions/predict-the-labour-market-status-of-participants'
train_data <- read.csv(file.path(data_dir, 'train.csv'), stringsAsFactors = FALSE)
test_data  <- read.csv(file.path(data_dir, 'test.csv'),  stringsAsFactors = FALSE)

cat('Training rows:', nrow(train_data), '| columns:', ncol(train_data), '\n')
cat('Testing rows :', nrow(test_data),  '| columns:', ncol(test_data),  '\n')

# The target column in train_data is `employed_status` (1 = employed, 0 = unemployed).
# It is NOT present in test_data — that is what you predict.


Warning message in file(file, "rt"):
“cannot open file '/kaggle/input/competitions/predict-the-labour-market-status-of-participants/train.csv': No such file or directory”


ERROR: Error in file(file, "rt"): cannot open the connection


## 📊 Step 2: Get to know the data
Before modelling, look at what the data actually contains. Each chart below answers one question a
first-time viewer should ask, and ends with **what it means for your model**.

> **Sizing tip:** on Kaggle/Jupyter, control chart size with `options(repr.plot.width = ..., repr.plot.height = ...)`
> *before* the plot — not with fixed panel sizes. We set sensible sizes per chart below so nothing gets cropped.


In [ ]:
# A tidy plotting frame with readable labels
plot_df <- train_data %>%
  filter(!is.na(employed_status)) %>%
  mutate(Status = factor(employed_status, levels = c(0, 1),
                         labels = c('Unemployed', 'Employed')))

pal <- c('Unemployed' = '#E15759', 'Employed' = '#59A14F')

# A reusable, clean theme
theme_hack <- theme_minimal(base_size = 13) +
  theme(plot.title = element_text(face = 'bold'),
        plot.subtitle = element_text(color = 'grey35'),
        legend.position = 'top')


### 2.1 — How many people are employed? (class balance)
The single most important thing to know before modelling: what fraction of the training set is employed?

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4)

rate <- mean(plot_df$Status == 'Employed')

ggplot(plot_df, aes(Status, fill = Status)) +
  geom_bar(color = 'grey20', width = 0.7) +
  geom_text(stat = 'count', aes(label = comma(after_stat(count))), vjust = -0.4, size = 4.5) +
  scale_fill_manual(values = pal) +
  scale_y_continuous(labels = comma, expand = expansion(mult = c(0, 0.12))) +
  labs(title = 'Most people in the data are unemployed',
       subtitle = sprintf('Overall employment rate: %.1f%%', 100 * rate),
       x = NULL, y = 'People') +
  theme_hack + theme(legend.position = 'none')


**What it means:** the classes are imbalanced (~1 in 3 employed). A model that predicts 'unemployed' for everyone would look ~68% 'accurate' but is useless — this is exactly why the competition uses **AUC**, which rewards ranking people by their probability of employment rather than raw accuracy.

### 2.2 — Who has history, and who is brand new?
Most of the 'lag' (previous-round) columns are missing. That is not dirty data — it is because many people appear for the *first* time in their row, so they have no earlier round to look back on.

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4)

miss <- train_data %>%
  summarise(across(everything(), ~ mean(is.na(.)))) %>%
  tidyr::pivot_longer(everything(), names_to = 'column', values_to = 'pct_missing') %>%
  arrange(desc(pct_missing)) %>%
  slice_head(n = 12) %>%
  mutate(column = factor(column, levels = rev(column)))   # lock labels to sorted bars

ggplot(miss, aes(column, pct_missing)) +
  geom_col(fill = '#4E79A7', color = 'grey20') +
  coord_flip() +
  scale_y_continuous(labels = percent) +
  labs(title = 'Which columns are mostly empty?',
       subtitle = 'Subject marks, SETA, and tertiary fields are the sparsest',
       x = NULL, y = 'Share missing') +
  theme_hack + theme(legend.position = 'none')


In [ ]:
# How many prior observations does each person have?
options(repr.plot.width = 7, repr.plot.height = 4)

ggplot(train_data, aes(factor(total_historical_rounds))) +
  geom_bar(fill = '#8CD17D', color = 'grey20') +
  labs(title = 'Most people are first-timers',
       subtitle = 'Count of prior survey rounds per person',
       x = 'Total historical rounds', y = 'People') +
  theme_hack


**What it means:** the lag columns are powerful *when present* but empty for most people. Decide deliberately how to handle those NAs — dropping them throws away most of your data. A common trick (used in Step 5) is to treat 'missing history' as its own category rather than discarding the row.

### 2.3 — Does last round's status predict this round? (the key signal)
For people who *do* have history, their previous labour-market status is the strongest simple predictor there is.

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4.5)

hist_data <- plot_df %>% filter(!is.na(status_broad_lag))

ggplot(hist_data, aes(x = reorder(status_broad_lag, employed_status, mean), fill = Status)) +
  geom_bar(position = 'fill', color = 'grey20') +
  coord_flip() +
  scale_y_continuous(labels = percent) +
  scale_fill_manual(values = pal) +
  labs(title = 'Your past status strongly shapes your present',
       subtitle = 'Current employment rate, grouped by status in the PREVIOUS round',
       x = 'Status in previous round', y = 'Share now employed') +
  theme_hack


**What it means:** someone employed last round is more likely to be employed now than someone who was unemployed. This 'stickiness' is part of the story — but as you'll see, it is not the whole picture, and turning it into leaderboard gains takes care.

### 2.4 — The gender gap
A demographic split that is large and consistent in this data.

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4)

gap <- plot_df %>% filter(!is.na(gender)) %>%
  group_by(gender) %>% summarise(rate = mean(employed_status), n = n())

ggplot(gap, aes(gender, rate, fill = gender)) +
  geom_col(color = 'grey20', width = 0.6) +
  geom_text(aes(label = percent(rate, accuracy = 0.1)), vjust = -0.4, size = 4.5) +
  scale_y_continuous(labels = percent, expand = expansion(mult = c(0, 0.12))) +
  labs(title = 'Employment rate differs by gender',
       x = NULL, y = 'Employment rate') +
  theme_hack + theme(legend.position = 'none')


### 2.5 — Does education move the needle?
Employment rate across the highest education levels present in the data.

In [ ]:
options(repr.plot.width = 9, repr.plot.height = 5)

edu <- plot_df %>% filter(!is.na(education_level)) %>%
  group_by(education_level) %>%
  summarise(rate = mean(employed_status), n = n()) %>%
  filter(n >= 50)   # ignore tiny, noisy groups

ggplot(edu, aes(reorder(education_level, rate), rate)) +
  geom_col(fill = '#B07AA1', color = 'grey20') +
  geom_text(aes(label = percent(rate, accuracy = 1)), hjust = -0.15, size = 3.5) +
  coord_flip() +
  scale_y_continuous(labels = percent, expand = expansion(mult = c(0, 0.15))) +
  labs(title = 'Employment rate by highest education level',
       subtitle = 'Groups with at least 50 people',
       x = NULL, y = 'Employment rate') +
  theme_hack + theme(axis.text.y = element_text(size = 9))


### 2.6 — Does the Work Readiness Score matter?
Harambee's behavioural assessment, compared between employed and unemployed people.

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4)

score_data <- plot_df %>% filter(!is.na(work_readiness_score))

ggplot(score_data, aes(Status, work_readiness_score, fill = Status)) +
  geom_boxplot(color = 'grey20', alpha = 0.8, outlier.alpha = 0.15) +
  scale_fill_manual(values = pal) +
  labs(title = 'Work Readiness Score vs employment',
       x = NULL, y = 'Work Readiness Score (0-1)') +
  theme_hack + theme(legend.position = 'none')


**What it means:** if the boxes overlap heavily, the score alone is a weak separator — and adding weak features to a model can even *hurt* it. Part of the challenge is finding which combinations genuinely carry signal.

## 🏗️ Step 3: A first baseline model
We predict a binary outcome, so we start with **logistic regression**. For this first attempt we deliberately
pick two features that turn out to carry little signal: `work_readiness_score` and `sa_citizen`.
This sets a (low) floor — and shows that simply throwing variables at the model is not enough.

In [ ]:
# Fill missing work readiness with the training median so every row has a value
wr_median <- median(train_data$work_readiness_score, na.rm = TRUE)
train_data$wr_filled <- ifelse(is.na(train_data$work_readiness_score), wr_median, train_data$work_readiness_score)
test_data$wr_filled  <- ifelse(is.na(test_data$work_readiness_score),  wr_median, test_data$work_readiness_score)

# Treat citizenship as a category
train_data$sa_citizen_chr <- as.character(train_data$sa_citizen)
test_data$sa_citizen_chr  <- as.character(test_data$sa_citizen)

baseline_model <- glm(employed_status ~ wr_filled + sa_citizen_chr,
                      data = train_data, family = binomial)

summary(baseline_model)


## 🔮 Step 4: Predict and build a submission
For AUC we submit **probabilities**, not hard labels. The block below writes a valid `submission_baseline.csv`.
Upload it on the competition's Submit page to see your public leaderboard score.

In [ ]:
# Make sure the test set has the same prepared column the model was trained on
test_data$sa_citizen_chr <- as.character(test_data$sa_citizen)

# Predicted PROBABILITY of employment (0-1)
test_prob <- predict(baseline_model, newdata = test_data, type = 'response')

submission <- data.frame(anonymised_id = test_data$anonymised_id,
                         employed_status = test_prob)
write.csv(submission, 'submission_baseline.csv', row.names = FALSE, quote = FALSE)
cat('Saved submission_baseline.csv with', nrow(submission), 'rows\n')
head(submission)


## 🔄 Step 5: Choose better features and watch the score improve
The baseline features barely helped. Now we swap in features that actually carry signal:
`gender` and `education_level`, while keeping `work_readiness_score`. Same model, better inputs — and the score improves.
This is the core lesson: **which** variables you choose matters more than how many.

In [ ]:
# Fill missing work readiness (recompute to be safe)
wr_median <- median(train_data$work_readiness_score, na.rm = TRUE)
train_data$wr_filled <- ifelse(is.na(train_data$work_readiness_score), wr_median, train_data$work_readiness_score)
test_data$wr_filled  <- ifelse(is.na(test_data$work_readiness_score),  wr_median, test_data$work_readiness_score)

# Align education_level factor levels between train and test
train_data$education_level <- factor(train_data$education_level)
test_data$education_level  <- factor(test_data$education_level, levels = levels(train_data$education_level))

upgraded_model <- glm(employed_status ~ gender + education_level + wr_filled,
                      data = train_data, family = binomial)

up_prob <- predict(upgraded_model, newdata = test_data, type = 'response')
up_prob[is.na(up_prob)] <- mean(train_data$employed_status, na.rm = TRUE)

submission2 <- data.frame(anonymised_id = test_data$anonymised_id, employed_status = up_prob)
write.csv(submission2, 'submission_upgraded.csv', row.names = FALSE, quote = FALSE)
cat('Saved submission_upgraded.csv\n')
head(submission2)


## 🚀 Over to you — ideas to climb the leaderboard
You just saw that swapping weak features for better ones lifted the score. There is a lot more room to improve:

- **Use participant history:** `employed_lag`, `status_lag`, `status_broad_lag`, `tenure_lag` — a person's *previous* status is a strong clue, when it is present.
- **Handle missingness thoughtfully:** treat 'missing' as its own category; train and test differ in how complete some columns are.
- **More demographics & geography:** `age`, `race`, `province`, `district`, `municipality`.
- **Education & skills:** `school_quintile`, the `matric_*` subjects.
- **Try stronger models:** `ranger` (random forest) or `xgboost`.
- **Always submit probabilities** and validate with AUC, not accuracy.

Good luck!